In [1]:
import argparse
import os
import time
import datetime
import torch
import torch.optim as optim
from math import ceil
from torch.utils.data import DataLoader
from dataset.dataset import CityScapesDataset
from utils.metric import runningScore, averageMeter
import numpy as np
from tqdm import tqdm



os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
torch.cuda.is_available()

True

In [3]:
# n_train=2436
n_train=2476
n_val=400


batchsize   = 8 
Epoch       = 80
img_size    = [256, 512]
model_name  = 'model_task1'
task        = 'cat'
lr = 1e-3 

final_save_path='./'+model_name+'.pth'

In [4]:
def train(epoch, data_loader, Net, optimizer, loss_fn, Meter):
    Net.train()
    timeStart = time.time()
    with tqdm(total=n_train, desc=f'Epoch {epoch + 1}/{Epoch}', unit='img') as pbar:
        for i, (data, target) in enumerate(data_loader):
            data , target = data.to(device),target.to(device)
            ##yourself
            optimizer.zero_grad()
            pred = Net(data)                    # B x C x H x W
            loss = loss_fn(pred, target)        # CE loss (已經是 scalar)

            loss.backward()
            optimizer.step()
            
            training_loss = loss.item()
            pbar.set_postfix(**{'loss (batch)': training_loss})
            pred = pred.data.max(1)[1]
            Meter['metric'].update(target.data.cpu().numpy(), pred.data.cpu().numpy())
            Meter['loss'].update(training_loss,data.size()[0])
            pbar.update(data.shape[0])
    timeEnd = time.time()
    score, class_iou = Meter['metric'].get_scores()
    loss_avg = Meter['loss'].avg
    print('epoch %3d : %10s loss: %f OverallAcc: %f MeanAcc %f mIoU %f time: %f' 
        %(epoch, ('training'), loss_avg, score['OverallAcc'], score['MeanAcc'], score['mIoU'], timeEnd-timeStart))

    return score['mIoU'],loss_avg

In [5]:
def val(epoch, data_loader, Net, loss_fn, Meter):
    Net.eval()
    with torch.no_grad():
        for i, (data, target) in enumerate(data_loader):
            data, target = data.to(device), target.to(device)
            timeStart = time.time()
            ### evaluate by yourself
            pred = Net(data)
            validation_loss = loss_fn(pred, target)
            
            timeEnd = time.time()           
            
            pred = pred.data.max(1)[1]
            Meter['metric'].update(target.data.cpu().numpy(), pred.data.cpu().numpy())
            Meter['loss'].update(validation_loss,data.size()[0])
            Meter['time'].update(timeEnd-timeStart,1)
    score, class_iou = Meter['metric'].get_scores()
    loss_avg = Meter['loss'].avg
    time_avg = Meter['time'].avg
    print('epoch %3d : %10s loss: %f OverallAcc: %f MeanAcc %f mIoU %f time: %f' 
        %(epoch, ('validation'), loss_avg, score['OverallAcc'], score['MeanAcc'], score['mIoU'], time_avg))
    
    return score['mIoU']

In [6]:
best_val_miou=0
current_val_miou=0

In [7]:
import network
#from network import *
if __name__ == '__main__':

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
    assert task in ['cat'],'wrong value of task'
    if task=='cat':
        num_classes = 8
        
    training_meter = {'metric':runningScore(num_classes),'loss':averageMeter(),'time':averageMeter()}
    validation_meter = {'metric':runningScore(num_classes),'loss':averageMeter(),'time':averageMeter()}

    print(str(datetime.datetime.now()))
    print('batchsize %3d | epoch %3d | img_size  %4d %4d | task  %6s | model_name  %25s '
                    %(batchsize,Epoch,img_size[0],img_size[1], task ,model_name ))

    TrainingDataset   = CityScapesDataset("data", "training", img_size, task=task, augmentation=None)
    ValidationDataset = CityScapesDataset("data", "validation", img_size, task=task)

    TrainingLoader    = DataLoader(TrainingDataset, batch_size=batchsize, shuffle=True, num_workers=4)
    ValidationLoader  = DataLoader(ValidationDataset, batch_size=batchsize, shuffle=False, num_workers=4)
    num_batch         = ceil(len(TrainingDataset)/batchsize)

    # define yout model
    Net = network.UNet(n_channels=3, n_classes=num_classes)
    Net = Net.to(device)
    # define your optimizer
    optimizer = optim.Adam(
        Net.parameters(),
        lr=lr,
        weight_decay=1e-4
    )
    
    # scheduler
    scheduler = optim.lr_scheduler.StepLR(
        optimizer,
        step_size=20,   # 每 20 個 epoch 降一次 lr
        gamma=0.5       # lr *= 0.5
    )
    # define your loss   
    loss_fn = torch.nn.CrossEntropyLoss(ignore_index=255)

    start_epoch    = 0
    best_val_miou  = 0.0
    best_state_dict = None
       
    for epoch in range(start_epoch, Epoch):
        for _, v in training_meter.items():
            v.reset()
            
        show_lr=optimizer.param_groups[0]['lr']
        print('learning rate is : ',  show_lr)
        current_train_miou,current_train_loss=train(epoch, TrainingLoader, Net, optimizer, loss_fn, training_meter)
        
        scheduler.step() ###  StepLR
        
        if (epoch+1)%1==0 or epoch==Epoch-1:
            for _, v in validation_meter.items():
                v.reset()
            current_val_miou=val(epoch, ValidationLoader, Net, loss_fn, validation_meter)
            
            if current_val_miou>best_val_miou:
                best_val_miou=current_val_miou
                best_state_dict = Net.state_dict()
                torch.save(best_state_dict, final_save_path)
                print("(model saved)")
    print(str(datetime.datetime.now()))


2025-11-15 12:07:13.198746
batchsize   8 | epoch  80 | img_size   256  512 | task     cat | model_name                model_task1 
learning rate is :  0.001


Epoch 1/80: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 2476/2476 [02:28<00:00, 16.68img/s, loss (batch)=0.421]

epoch   0 :   training loss: 0.684172 OverallAcc: 0.788134 MeanAcc 0.568575 mIoU 0.461262 time: 148.443029


epoch   0 : validation loss: 0.823498 OverallAcc: 0.705980 MeanAcc 0.567734 mIoU 0.436314 time: 0.002945
(model saved)
learning rate is :  0.001


Epoch 2/80: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 2476/2476 [02:24<00:00, 17.13img/s, loss (batch)=0.59]

...

epoch  78 :   training loss: 0.086522 OverallAcc: 0.969164 MeanAcc 0.928869 mIoU 0.881465 time: 144.783945


epoch  78 : validation loss: 0.320875 OverallAcc: 0.922895 MeanAcc 0.857959 mIoU 0.786634 time: 0.002868
learning rate is :  0.000125


Epoch 80/80: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 2476/2476 [02:22<00:00, 17.39img/s, loss (batch)=0.0792]

epoch  79 :   training loss: 0.083405 OverallAcc: 0.970223 MeanAcc 0.931035 mIoU 0.884951 time: 142.376503


epoch  79 : validation loss: 0.317005 OverallAcc: 0.924787 MeanAcc 0.868961 mIoU 0.793491 time: 0.002691
2025-11-15 15:54:11.758600


In [8]:
print(' best_val_miou is :  ',best_val_miou)

 best_val_miou is :   0.7977337594711459
